# Healthcare Data Analysis Using Python
## Exploratory Data Analysis of Hospital Distribution in India

**Dataset:** `hospital_directory.csv`  
**Source:** Government of India Open Government Data Platform (data.gov.in)  
**Objective:** Explore hospital distribution across states and districts, hospital categories and care types, available bed information, and data quality.

> **Important:** This notebook analyzes the dataset as provided. A value of `0` may mean “not reported” for some fields, so zeros are not automatically treated as missing in every column. Interpret results as patterns in this directory, not as proof of hospital quality or patient outcomes.

### Project checklist
- Load and inspect the dataset
- Check shape, data types, missing values, and duplicates
- Clean column names and selected placeholder values
- Create derived columns where useful
- Perform univariate, bivariate, and multivariate analysis
- Use groupby, pivot table, and correlation analysis
- Create at least 10 visualizations
- Summarize at least 5 findings supported by calculated results


## Problem Statement
**Domain:** Public healthcare in India.

**Problem:** Hospital information in India is spread across thousands of directory entries. It is unclear how evenly hospitals are distributed across states and regions, and how complete the information in the directory is.

**Questions I want to answer:**
1. Which states and regions have the most hospital records?
2. How many records say whether a hospital is private or public, and what care type it is?
3. How many hospitals publish contact details that a patient could use?
4. Does data quality differ between regions, or between private and public hospitals?

**Why this dataset:** It comes from the Government of India open data portal (data.gov.in). It has 30,273 records and 48 columns, well above the 500 records and 10 columns required. It mixes categories, numbers, text and map coordinates.


## 1. Import libraries

We use Pandas for data handling, NumPy for numerical operations, Matplotlib and Seaborn for visualizations, and SciPy for statistical tests. If an import fails, install the missing package in the VS Code terminal using `python -m pip install pandas numpy matplotlib seaborn scipy jupyter`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
from scipy import stats

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11

print("Libraries imported successfully.")

## 2. Load the original CSV file

Place `hospital_directory.csv` in the same folder as this notebook. The relative path below helps the notebook work on another computer without a hard-coded personal file path.


In [ ]:
file_path = Path("hospital_directory.csv")

if not file_path.exists():
    raise FileNotFoundError(
        "hospital_directory.csv was not found. Put the CSV in the same folder as this notebook."
    )

df_raw = pd.read_csv(file_path, low_memory=False)
df = df_raw.copy()

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
display(df.head())

## 3. Initial overview

`head()` shows sample records, `info()` lists data types and non-null counts, and `describe()` summarizes numeric columns. We inspect before cleaning so the original structure is understood.


In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nDataFrame information:")
df.info()

print("\nNumeric summary:")
display(df.describe(include="number").T)

print("\nSummary including text columns:")
display(df.describe(include="all").T)

## 4. Data quality checks

Check exact duplicate rows, missing values, and placeholder zeros. In this directory, `0` may be used as a placeholder in some fields, but it may be a legitimate value in others. We will not globally replace every zero.


In [ ]:
duplicate_count = int(df.duplicated().sum())
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)

print("Exact duplicate rows:", duplicate_count)
display(missing_summary)

# Count zero-like values in selected fields where zero often means not provided.
placeholder_fields = [
    "Hospital_Category", "Hospital_Care_Type", "Subdistrict", "Telephone",
    "Mobile_Number", "Emergency_Num", "Website", "Specialties", "Facilities",
    "Total_Num_Beds", "Number_Doctor", "Establised_Year"
]
available_placeholder_fields = [c for c in placeholder_fields if c in df.columns]
zero_summary = {}
for col in available_placeholder_fields:
    zero_summary[col] = int((df[col].astype(str).str.strip() == "0").sum())
print("Rows with literal '0' in selected fields:")
display(pd.Series(zero_summary, name="zero_placeholder_count").to_frame())

## 5. Clean and prepare the data

We standardize column names for easier coding, trim extra spaces in text fields, and remove exact duplicate rows only if any exist. We keep the raw dataframe (`df_raw`) unchanged. For selected descriptive fields, a literal `0` is treated as “not reported” because it is not a meaningful category/name; we do not apply that conversion to every column. Numeric fields are converted carefully.


In [ ]:
# Make column names easier to type while retaining their meaning.
df.columns = (
    df.columns.str.strip()
      .str.lower()
      .str.replace(r"[^a-z0-9]+", "_", regex=True)
      .str.strip("_")
)

# Trim whitespace in text columns.
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].astype("string").str.strip()

# Remove exact duplicate rows if present.
before_rows = len(df)
df = df.drop_duplicates().copy()
print("Rows removed as exact duplicates:", before_rows - len(df))

# In these text fields, the literal value '0' is treated as not reported.
text_placeholder_cols = [
    "hospital_name", "hospital_category", "hospital_care_type",
    "discipline_systems_of_medicine", "state", "district",
    "specialties", "facilities", "website", "town", "subtown", "village"
]
for col in text_placeholder_cols:
    if col in df.columns:
        df[col] = df[col].replace({"0": pd.NA, "": pd.NA, "nan": pd.NA})

# Convert selected numeric columns to numeric values.
numeric_candidates = [
    "total_num_beds", "number_doctor", "num_mediconsultant_or_expert",
    "number_private_wards", "num_bed_for_eco_weaker_sec", "establised_year",
    "state_id", "district_id"
]
for col in numeric_candidates:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("Cleaned shape:", df.shape)
display(df.head())

## 5b. Extra cleaning: fixing problems found during the quality checks

The overview showed that about 93% of rows carry `0` in most descriptive columns, and a few values are clearly wrong. These fixes are applied on the cleaned copy `df` (the raw data `df_raw` stays unchanged):

1. **Contact fields** – a literal `0` in telephone, mobile, e-mail and similar fields means "not provided", so it becomes missing.
2. **Wrong state name** – one record has a *district* (North Twenty Four Parganas) typed in the State column. For records with a valid `state_id`, we replace such a rare, invalid name with the most common state name for that ID. `state_id = 0` is a catch-all code shared by records from many different states, so it is never used for fixing (doing so would wrongly relabel them all as Himachal Pradesh).
3. **Care type spelling** – variants such as `Medical College / Institute/ Hospital` are merged into one group.
4. **Pincode** – a valid Indian pincode has 6 digits and does not start with 0; anything else becomes missing.

In [ ]:
# 1. Zero placeholders in contact / registration style fields
contact_cols = ["telephone", "mobile_number", "emergency_num", "ambulance_phone_no", "bloodbank_phone_no",
                "tollfree", "helpline", "hospital_fax", "hospital_primary_email_id",
                "hospital_secondary_email_id", "address_original_first_line", "location"]
for col in contact_cols:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()
        df[col] = df[col].mask(df[col].isin(["0", "", "nan", "NA"]))

# 2. Fix the wrong State value using state_id (each id must map to one state)
# state_id 0 is a catch-all code used by ~340 records from many states, so it is NOT used for fixing.
# Only rare, invalid "state" names (e.g. a district typed in the State column) are replaced.
state_counts = df["state"].value_counts()
rare = df["state"].map(state_counts) < 5
state_by_id = df[df["state_id"] > 0].groupby("state_id")["state"].agg(lambda s: s.mode().iat[0])
fix = rare & df["state_id"].map(state_by_id).notna()
print("State values corrected using state_id:", int(fix.sum()))
df.loc[fix, "state"] = df.loc[fix, "state_id"].map(state_by_id)
print("Distinct states/UTs now:", df["state"].nunique())

# 3. Merge spelling variants of care type
df["hospital_care_type"] = df["hospital_care_type"].str.replace(
    r"^Medical College / Institute.*$", "Medical College / Institute", regex=True)

# 4. Valid pincodes only
valid_pin = df["pincode"].astype("string").str.strip().str.fullmatch(r"[1-9]\d{5}").fillna(False)
print("Invalid / missing pincodes set to missing:", int((~valid_pin).sum()))
df["pincode"] = df["pincode"].astype("string").str.strip().where(valid_pin)

### Drop columns that contain only zeros
These 11 columns contain only `0` for every record, so they hold no information. They are removed from the working copy `df`. The raw data `df_raw` and the original CSV are unchanged. Rows are **not** deleted, because most records have `0` in many fields and removing them would discard almost all of the data.


In [ ]:
all_zero_cols = ["accreditation", "hospital_regis_number", "registeration_number_scan", "subtown", "village", "miscellaneous_facilities", "num_mediconsultant_or_expert", "number_private_wards", "num_bed_for_eco_weaker_sec", "empanelment_or_collaboration_with", "tariff_range"]
all_zero_cols = [c for c in all_zero_cols if c in df.columns and (df[c].fillna(0) == 0).all()]
df = df.drop(columns=all_zero_cols)
print("Dropped", len(all_zero_cols), "columns:", all_zero_cols)
print("Shape now:", df.shape)

## 6. Create useful derived columns

Derived columns turn raw fields into things we can analyse:

* `state_clean`, `district_clean` – tidy text.
* `reported_beds` – bed counts that are plausible (1 to 5,000). Zero means "not reported", and a few huge values (billions of beds) are obvious data-entry errors, so they are removed.
* `latitude`, `longitude` – split from the coordinates text and checked to be inside India.
* `has_phone`, `has_mobile`, `has_email`, `has_website`, `has_coordinates` – 1/0 flags for whether the contact detail exists, and `contact_score` (0–4) which adds the first four.
* `region` – North / South / East / West / Central / North-East group of the state.
* `facility_type` – the kind of facility guessed from keywords in the hospital name (e.g. Clinic, Nursing Home, Eye/Dental).
* `n_specialties`, `n_facilities` – how many specialties / facilities are listed.
* `pincode_zone` – first digit of the pincode (India Post region 1–9).

In [ ]:
df["state_clean"] = df["state"].str.strip().str.title()
df["district_clean"] = df["district"].str.strip().str.title()

# Plausible bed counts only
raw_beds = df["total_num_beds"]
df["reported_beds"] = raw_beds.where(raw_beds.between(1, 5000))
print("Records with a bed value:", int((raw_beds > 0).sum()),
      "| kept as plausible:", int(df["reported_beds"].notna().sum()),
      "| removed as impossible:", int((raw_beds > 5000).sum()))

# Coordinates -> latitude / longitude (kept only if inside India's bounding box)
coords = df["location_coordinates"].str.split(",", expand=True)
df["latitude"] = pd.to_numeric(coords[0], errors="coerce")
df["longitude"] = pd.to_numeric(coords[1], errors="coerce")
inside = df["latitude"].between(6, 38) & df["longitude"].between(68, 98)
df.loc[~inside, ["latitude", "longitude"]] = np.nan
print("Records with usable coordinates:", int(df["latitude"].notna().sum()))

# Contact availability flags
df["has_phone"] = df["telephone"].notna().astype(int)
df["has_mobile"] = df["mobile_number"].notna().astype(int)
df["has_email"] = df["hospital_primary_email_id"].notna().astype(int)
df["has_website"] = df["website"].notna().astype(int)
df["has_coordinates"] = df["latitude"].notna().astype(int)
df["contact_score"] = df[["has_phone", "has_mobile", "has_email", "has_website"]].sum(axis=1)

# Region of the state
region_map = {
    "North": ["Jammu And Kashmir", "Himachal Pradesh", "Punjab", "Haryana", "Uttarakhand", "Chandigarh", "Uttar Pradesh"],
    "West": ["Rajasthan", "Gujarat", "Maharashtra", "Goa", "Dadra And Nagar Haveli", "Daman And Diu"],
    "Central": ["Madhya Pradesh", "Chhattisgarh"],
    "South": ["Andhra Pradesh", "Telangana", "Karnataka", "Tamil Nadu", "Kerala", "Puducherry", "Lakshadweep"],
    "East": ["West Bengal", "Odisha", "Bihar", "Jharkhand", "Andaman And Nicobar Islands"],
    "North-East": ["Assam", "Meghalaya", "Mizoram", "Manipur", "Nagaland", "Tripura", "Arunachal Pradesh", "Sikkim"],
}
state_to_region = {s: r for r, lst in region_map.items() for s in lst}
df["region"] = df["state_clean"].map(state_to_region)
print("States not assigned to a region:", df.loc[df["region"].isna(), "state_clean"].unique().tolist())

# Facility type guessed from keywords in the name (first matching rule wins)
rules = [
    ("Medical college / teaching", r"medical college|institute of medical|aiims|medical sciences|university"),
    ("Ayush / traditional", r"ayurved|homeo|homoeo|unani|siddha|naturopath|ayush"),
    ("Nursing home / maternity", r"nursing home|maternity|matern"),
    ("Eye / dental / ENT", r"\beye\b|netralaya|ophthal|dental|\bent\b"),
    ("Child / women / fertility", r"child|pediatric|paediatric|women|mahila|gynae|gynec|ivf|fertility"),
    ("Heart / ortho / cancer / other speciality", r"cardio|heart|ortho|cancer|onco|neuro|kidney|nephro|urolog|trauma|spine|diabet"),
    ("Diagnostic / lab", r"diagnostic|laborator|imaging|scan|pathology"),
    ("Clinic / dispensary", r"clinic|polyclinic|dispensary"),
    ("Government / community health", r"civil hospital|district hospital|community health|primary health|general hospital|government|govt|\besi\b"),
    ("Multi / super speciality", r"multi|super|speciality|specialty|specialist"),
    ("General hospital", r"hospital"),
]
name_lower = df["hospital_name"].str.lower()
df["facility_type"] = "Other"
assigned = pd.Series(False, index=df.index)
for label, pattern in rules:
    hit = name_lower.str.contains(pattern, regex=True, na=False) & ~assigned
    df.loc[hit, "facility_type"] = label
    assigned |= hit

# Counts of listed specialties / facilities
def count_items(s):
    return s.fillna("").str.split(r"[,\n]+").apply(lambda x: len([i for i in x if i.strip()]))
df["n_specialties"] = count_items(df["specialties"])
df["n_facilities"] = count_items(df["facilities"])

# Pincode zone (first digit)
df["pincode_zone"] = df["pincode"].str[0]

df[["state_clean", "region", "facility_type", "contact_score", "latitude", "longitude", "reported_beds"]].head()

## 7. Cleaned data summary

Review how many values remain missing after the targeted cleaning. Missing values are not automatically filled with zero because that could create misleading results.


In [ ]:
clean_missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)
display(clean_missing.head(20))

print(f"Records available for analysis: {len(df):,}")
print(f"Columns available for analysis: {len(df.columns)}")

### Cleaning summary
| Step | What was found | What was done |
|---|---|---|
| Duplicate rows | 0 | Nothing to remove |
| Zero placeholders | For example 28,233 records have `0` in Hospital_Category | Changed `0` to missing in descriptive and contact fields (not filled with 0) |
| Wrong state names | 3 records | Fixed using `state_id` (the catch-all `state_id = 0` is never used) |
| Invalid pincodes | 408 | Set to missing |
| Impossible bed counts | 2 values above 5,000 | Removed from `reported_beds` |
| Columns with only zeros | 11 columns | Dropped from the working copy |
| Map coordinates | 10,672 usable records | Split into latitude and longitude and checked to be inside India |
| New columns | region, facility_type, contact_score, has_phone, has_email, has_website and more | Created as derived columns |

Missing values were **not** filled with 0, because that would create misleading results.


## Types of analysis used
| Type | Visualizations |
|---|---|
| Univariate | 1 (state), 2 (category), 3 (care type), 5 and 6 (beds), 7 (contacts), 9 (facility type), 11 (districts), 13 (completeness) |
| Bivariate | 4 (districts within the top state), 12 (care type by state), 15 (contact score by region) |
| Multivariate | 8 (map by region), 10 (correlation heatmap), 14 (state by data-field heatmap) |


## 8. Visualization 1: hospital records by state

This chart shows the number of directory records in the 15 states/UTs with the most listed hospitals. It measures entries in this dataset, not necessarily the true total number of operating hospitals.

In [ ]:
if "state_clean" in df.columns:
    state_counts = df["state_clean"].dropna().value_counts().head(15)
    ax = state_counts.sort_values().plot(kind="barh", figsize=(10, 7))
    ax.set_title("Top 15 States/UTs by Hospital Directory Records")
    ax.set_xlabel("Number of records")
    ax.set_ylabel("State / Union Territory")
    plt.tight_layout()
    plt.show()
    display(state_counts.rename("hospital_records").to_frame())
else:
    print("State column not available.")

**Your interpretation:** *(write 2–3 sentences: which states lead, how big is the gap between the top and the rest, and why that might happen. Hint: bigger or more populated states should have more hospitals, but also check which states report data more completely.)*

## 9. Visualization 2: hospital category (private vs public)

Most records do not report a category, so "Not reported" is shown separately. Only the reported ones can be compared.

In [ ]:
if "hospital_category" in df.columns:
    category_counts = df["hospital_category"].fillna("Not reported").value_counts().head(12)
    ax = category_counts.sort_values().plot(kind="barh", figsize=(10, 6))
    ax.set_title("Hospital Records by Category")
    ax.set_xlabel("Number of records")
    ax.set_ylabel("Hospital category")
    plt.tight_layout()
    plt.show()
    display(category_counts.to_frame("records"))
else:
    print("Hospital category column not available.")

## 10. Visualization 3: hospital care type

Compare the number of records by care type. Values that are missing or not reported remain visible as a separate group.

In [ ]:
if "hospital_care_type" in df.columns:
    care_counts = df["hospital_care_type"].fillna("Not reported").value_counts().head(12)
    plt.figure(figsize=(9, 5))
    sns.barplot(x=care_counts.values, y=care_counts.index)
    plt.title("Hospital Records by Care Type")
    plt.xlabel("Number of records")
    plt.ylabel("Care type")
    plt.tight_layout()
    plt.show()
else:
    print("Hospital care type column not available.")

## 11. Visualization 4: districts in the most represented state

First identify the state with the most directory records, then show its 15 most represented districts.

In [ ]:
if {"state_clean", "district_clean"}.issubset(df.columns):
    state_counts_all = df["state_clean"].dropna().value_counts()
    if not state_counts_all.empty:
        top_state = state_counts_all.index[0]
        district_counts = (
            df.loc[df["state_clean"] == top_state, "district_clean"]
              .dropna().value_counts().head(15)
        )
        print("State/UT with most records:", top_state)
        ax = district_counts.sort_values().plot(kind="barh", figsize=(10, 7))
        ax.set_title(f"Top Districts by Hospital Directory Records — {top_state}")
        ax.set_xlabel("Number of records")
        ax.set_ylabel("District")
        plt.tight_layout()
        plt.show()
    else:
        print("No state values available.")
else:
    print("State or district column not available.")

## 12. Visualization 5: distribution of reported bed counts

Only plausible positive bed counts are included (zero means not reported and values above 5,000 were removed as data errors). A histogram shows the distribution. **Note that only a small number of hospitals report beds**, so this chart describes those hospitals only.

In [ ]:
if "reported_beds" in df.columns:
    beds = df["reported_beds"].dropna()
    if not beds.empty:
        plt.figure(figsize=(10, 5))
        sns.histplot(beds, bins=30, kde=True)
        plt.title("Distribution of Reported Hospital Bed Counts")
        plt.xlabel("Reported number of beds")
        plt.ylabel("Number of hospital records")
        plt.tight_layout()
        plt.show()
        display(beds.describe().to_frame("reported_beds").T)
    else:
        print("No positive bed counts are available.")
else:
    print("Bed count column not available.")

## 13. Visualization 6: box plot of reported bed counts

A box plot highlights the median, spread and large hospitals among those that reported beds.

In [ ]:
if "reported_beds" in df.columns:
    beds = df["reported_beds"].dropna()
    if not beds.empty:
        plt.figure(figsize=(10, 3))
        sns.boxplot(x=beds)
        plt.title("Box Plot of Reported Hospital Bed Counts")
        plt.xlabel("Reported number of beds")
        plt.tight_layout()
        plt.show()

## 14. Visualization 7: how many hospitals publish contact details?

Doctor counts are reported by only one hospital, so a doctor chart would be empty. Contact details are a much better measure of how useful the directory is to a patient: we compare how many records have a phone, mobile, e-mail, website and map location.

In [ ]:
flags = {"Telephone": "has_phone", "Mobile": "has_mobile", "E-mail": "has_email",
         "Website": "has_website", "Map coordinates": "has_coordinates"}
share = pd.Series({k: df[v].mean() * 100 for k, v in flags.items()}).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bars = axes[0].barh(share.index, share.values, color="#4C78A8")
axes[0].set_title("Share of records with each contact detail")
axes[0].set_xlabel("% of all hospital records")
axes[0].set_xlim(0, 100)
for b, v in zip(bars, share.values):
    axes[0].text(v + 1, b.get_y() + b.get_height() / 2, f"{v:.1f}%", va="center")

score_counts = df["contact_score"].value_counts().sort_index()
axes[1].bar(score_counts.index.astype(str), score_counts.values, color="#F58518")
axes[1].set_title("Contact score per hospital (phone + mobile + e-mail + website)")
axes[1].set_xlabel("Number of contact details available (0-4)")
axes[1].set_ylabel("Number of records")
for x, v in zip(range(len(score_counts)), score_counts.values):
    axes[1].text(x, v, f"{v:,}", ha="center", va="bottom")
plt.tight_layout(); plt.show()
print(f"Records with NO phone, mobile, e-mail or website: {(df['contact_score'] == 0).mean():.1%}")

**Your interpretation:** *(What share of hospitals can a patient actually contact? Which contact detail is most common? What does this say about the quality of the directory?)*

## 15. Visualization 8: where are the hospitals? (map-style scatter plot)

Latitude and longitude were extracted from the coordinates column. Plotting them gives a rough outline of India, coloured by region. Only records with usable coordinates are shown, so the picture is incomplete in areas where coordinates are missing.

In [ ]:
geo = df.dropna(subset=["latitude", "longitude", "region"])
palette = dict(zip(sorted(geo["region"].unique()), sns.color_palette("tab10", geo["region"].nunique())))

plt.figure(figsize=(8, 9))
sns.scatterplot(data=geo, x="longitude", y="latitude", hue="region", palette=palette, s=10, alpha=0.6, linewidth=0)
plt.title("Hospital locations in the directory (records with coordinates)")
plt.xlabel("Longitude"); plt.ylabel("Latitude")
plt.legend(title="Region", markerscale=2, frameon=False)
plt.tight_layout(); plt.show()
print(f"Records plotted: {len(geo):,} of {len(df):,} ({len(geo) / len(df):.1%})")

**Your interpretation:** *(Which parts of the country look dense and which look empty? Is an "empty" area really without hospitals, or just missing coordinates? Compare with Visualization 1.)*

## 16. Visualization 9: hospital types guessed from their names, and regions

The dataset has no clean "type" column for most hospitals, so we classify them from words in the name (see the rules in section 6). The right-hand chart shows how records are spread across regions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={"width_ratios": [1.6, 1]})
ft = df["facility_type"].value_counts().sort_values()
axes[0].barh(ft.index, ft.values, color="#54A24B")
axes[0].set_title("Hospital records by facility type (from name keywords)")
axes[0].set_xlabel("Number of records")
for y, v in enumerate(ft.values):
    axes[0].text(v, y, f" {v:,} ({v / len(df):.1%})", va="center", fontsize=9)
axes[0].set_xlim(0, ft.max() * 1.25)

rg = df["region"].value_counts()
axes[1].pie(rg.values, labels=rg.index, autopct="%1.1f%%", startangle=90, pctdistance=0.8,
            wedgeprops={"width": 0.45, "edgecolor": "white"})
axes[1].set_title("Share of records by region")
plt.tight_layout(); plt.show()

## 17. Visualization 10: correlation heatmap of data-availability flags

The numeric fields about doctors and wards are almost entirely empty, so a correlation of them is meaningless. Instead we correlate the 1/0 flags showing *which details a record has*. A high correlation means records that give one detail tend to give the other too (they were filled in by the same careful data provider). Correlation does not imply causation.

In [ ]:
df["has_category"] = df["hospital_category"].notna().astype(int)
df["has_specialties"] = df["specialties"].notna().astype(int)
df["has_facilities"] = df["facilities"].notna().astype(int)

flag_cols = ["has_phone", "has_mobile", "has_email", "has_website", "has_coordinates",
             "has_category", "has_specialties", "has_facilities"]
corr = df[flag_cols].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1, square=True)
plt.title("Correlation between data-availability flags")
plt.tight_layout(); plt.show()

**Your interpretation:** *(Which pairs are strongly related? Why might hospitals that list a website also list e-mail or specialties?)*

## 18. Visualization 11: top district names overall

This chart shows the 15 district names with the most directory records, across all states. District names can repeat across states, so a district-only label may combine different places.

In [ ]:
if "district_clean" in df.columns:
    top_districts = df["district_clean"].dropna().value_counts().head(15)
    ax = top_districts.sort_values().plot(kind="barh", figsize=(10, 7))
    ax.set_title("Top 15 District Names by Hospital Directory Records")
    ax.set_xlabel("Number of records")
    ax.set_ylabel("District name")
    plt.tight_layout()
    plt.show()
else:
    print("District column not available.")

## 19. Visualization 12: care type by state

A stacked bar chart compares care-type composition for the 8 states/UTs with the most directory records. Because most records have no care type, "Not reported" dominates; the coloured parts show only the hospitals that did report.

In [ ]:
if {"state_clean", "hospital_care_type"}.issubset(df.columns):
    top_states = df["state_clean"].dropna().value_counts().head(8).index
    cross = df[df["state_clean"].isin(top_states)].copy()
    cross["hospital_care_type"] = cross["hospital_care_type"].fillna("Not reported")
    common_types = cross["hospital_care_type"].value_counts().head(5).index
    cross["care_type_plot"] = cross["hospital_care_type"].where(
        cross["hospital_care_type"].isin(common_types), "Other / less common"
    )
    pivot = pd.crosstab(cross["state_clean"], cross["care_type_plot"])
    pivot.plot(kind="bar", stacked=True, figsize=(12, 6))
    plt.title("Hospital Care Type by State/UT (Top 8 by Record Count)")
    plt.xlabel("State / Union Territory")
    plt.ylabel("Number of directory records")
    plt.xticks(rotation=35, ha="right")
    plt.legend(title="Care type", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()
else:
    print("State and care type columns are required for this chart.")

## 20. Visualization 13: completeness of selected fields

A completeness chart shows which fields have the most recorded information. This is a data-quality visualization, not a measure of hospital performance.

In [ ]:
# "Complete" means the field holds a real value (not missing, not a 0 placeholder, not an impossible value)
doctors_clean = pd.to_numeric(df["number_doctor"], errors="coerce").where(lambda s: s > 0)
completeness = pd.Series({
    "state": df["state"].notna().mean(),
    "district": df["district"].notna().mean(),
    "hospital_name": df["hospital_name"].notna().mean(),
    "pincode (valid)": df["pincode"].notna().mean(),
    "location_coordinates": df["has_coordinates"].mean(),
    "hospital_category": df["hospital_category"].notna().mean(),
    "hospital_care_type": df["hospital_care_type"].notna().mean(),
    "discipline_systems_of_medicine": df["discipline_systems_of_medicine"].notna().mean(),
    "telephone": df["has_phone"].mean(),
    "website": df["has_website"].mean(),
    "e-mail": df["has_email"].mean(),
    "specialties": df["specialties"].notna().mean(),
    "facilities": df["facilities"].notna().mean(),
    "reported_beds (plausible)": df["reported_beds"].notna().mean(),
    "number_doctor (> 0)": doctors_clean.notna().mean(),
}).mul(100).sort_values()

ax = completeness.plot(kind="barh", figsize=(10, 7), color="#4C78A8")
ax.set_title("Completeness of selected dataset fields (after cleaning)")
ax.set_xlabel("Records with a real value (%)")
ax.set_ylabel("Field")
ax.set_xlim(0, 105)
for y, v in enumerate(completeness.values):
    ax.text(v + 0.8, y, f"{v:.1f}%", va="center", fontsize=9)
plt.tight_layout(); plt.show()
display(completeness.rename("completeness_percent").round(2).to_frame())

## 21. Visualization 14: which states have the most complete records?

Multivariate view: for the 15 states with the most records, the heatmap shows what percentage of their hospitals have each detail. States where the colours are darker publish better data, which can distort the state-wise record counts in Visualization 1.

In [ ]:
top15 = df["state_clean"].value_counts().head(15).index
fields = {"Phone": "has_phone", "Mobile": "has_mobile", "E-mail": "has_email", "Website": "has_website",
          "Coordinates": "has_coordinates", "Category": "has_category", "Specialties": "has_specialties"}
comp_state = (df[df["state_clean"].isin(top15)].groupby("state_clean")[list(fields.values())].mean() * 100)
comp_state.columns = list(fields.keys())
comp_state = comp_state.loc[top15]

plt.figure(figsize=(11, 7))
sns.heatmap(comp_state, annot=True, fmt=".0f", cmap="YlGnBu", cbar_kws={"label": "% of the state's records"})
plt.title("Data completeness by state (top 15 states by number of records)")
plt.xlabel(""); plt.ylabel("")
plt.tight_layout(); plt.show()

## 22. Visualization 15: contact score by region (box / bar plot)

Does the number of available contact details differ by region? The bars show the average contact score and the line shows the percentage of hospitals with no contact detail at all.

In [ ]:
reg = (df.groupby("region")
         .agg(avg_contact_score=("contact_score", "mean"),
              pct_no_contact=("contact_score", lambda s: (s == 0).mean() * 100),
              records=("contact_score", "size"))
         .sort_values("avg_contact_score", ascending=False))
display(reg.round(2))

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(reg.index, reg["avg_contact_score"], color="#4C78A8")
ax1.set_ylabel("Average contact score (0-4)")
ax1.set_title("Contact details by region")
ax2 = ax1.twinx()
ax2.plot(reg.index, reg["pct_no_contact"], color="#D1495B", marker="o", lw=2)
ax2.set_ylabel("% of hospitals with no contact detail", color="#D1495B")
ax2.set_ylim(0, 100)
ax2.spines["right"].set_visible(True)
plt.tight_layout(); plt.show()

## 23. Groupby and pivot-table analysis

Groupby summarizes records by state and care type. A pivot table shows counts of records for the most represented states and care types.


In [ ]:
if {"state_clean", "hospital_care_type"}.issubset(df.columns):
    group_summary = (
        df.groupby("state_clean", dropna=True)
          .agg(
              directory_records=("state_clean", "size"),
              districts=("district_clean", "nunique") if "district_clean" in df.columns else ("state_clean", "size"),
              median_reported_beds=("reported_beds", "median") if "reported_beds" in df.columns else ("state_clean", "size")
          )
          .sort_values("directory_records", ascending=False)
    )
    display(group_summary.head(15))

    top_state_names = df["state_clean"].dropna().value_counts().head(10).index
    pivot_table = pd.pivot_table(
        df[df["state_clean"].isin(top_state_names)],
        index="state_clean",
        columns="hospital_care_type",
        values="hospital_name",
        aggfunc="count",
        fill_value=0,
        margins=True
    )
    display(pivot_table)
else:
    print("Required columns for groupby/pivot analysis are unavailable.")

### Pivot table: region × facility type

A second pivot table shows what share of each region's records falls into each facility type (rows add up to 100%).

In [ ]:
pt = pd.crosstab(df["region"], df["facility_type"], normalize="index") * 100
display(pt.round(1))

plt.figure(figsize=(13, 5))
sns.heatmap(pt, annot=True, fmt=".0f", cmap="Purples", cbar_kws={"label": "% of region's records"})
plt.title("Facility type mix by region (%)")
plt.xlabel(""); plt.ylabel("")
plt.xticks(rotation=40, ha="right")
plt.tight_layout(); plt.show()

## 24. Statistical summaries

Summary statistics for the numeric columns that actually have data: reported beds (positive plausible values only), number of listed specialties and facilities (only hospitals that list at least one), contact score and coordinates.

In [ ]:
summary = pd.DataFrame({
    "reported_beds": df["reported_beds"],
    "n_specialties (listed only)": df["n_specialties"].where(df["n_specialties"] > 0),
    "n_facilities (listed only)": df["n_facilities"].where(df["n_facilities"] > 0),
    "contact_score": df["contact_score"],
    "latitude": df["latitude"],
    "longitude": df["longitude"],
}).describe().T
summary["skew"] = pd.DataFrame({
    "reported_beds": df["reported_beds"], "n_specialties": df["n_specialties"].where(df["n_specialties"] > 0),
    "n_facilities": df["n_facilities"].where(df["n_facilities"] > 0), "contact_score": df["contact_score"],
    "latitude": df["latitude"], "longitude": df["longitude"]}).skew().values
display(summary.round(2))

## 25. Statistical tests

Charts suggest differences; tests check whether they are larger than chance. We use a 5% significance level (p < 0.05). With a large dataset, tiny differences can be "significant", so we also report an effect size (Cramér's V for chi-square tests; values near 0.1 are small, 0.3 medium, 0.5 large).

* **Chi-square test of independence** – are two categorical variables related?
* **Kruskal-Wallis test** – do three or more groups differ in a numeric/ordinal variable, without assuming a normal distribution?
* **Mann-Whitney U test** – same for two groups.

In [ ]:
from scipy import stats

def cramers_v(table):
    chi2 = stats.chi2_contingency(table)[0]
    n = table.values.sum()
    k = min(table.shape) - 1
    return np.sqrt(chi2 / (n * k))

def fmt_p(p):
    return "< 0.001" if p < 0.001 else f"{p:.3f}"

def verdict(p):
    return "significant" if p < 0.05 else "NOT significant"

rows = []

# 1. Region vs having a phone number
t = pd.crosstab(df["region"], df["has_phone"])
chi2, p, _, _ = stats.chi2_contingency(t)
rows.append(("Region vs has phone (chi-square)", f"chi2={chi2:.1f}, Cramer's V={cramers_v(t):.3f}", p))

# 2. Region vs contact score
groups = [g["contact_score"].values for _, g in df.groupby("region")]
H, p = stats.kruskal(*groups)
rows.append(("Contact score differs by region (Kruskal-Wallis)", f"H={H:.1f}", p))

# 3. Private vs public: website availability (only records that report a category)
cat = df[df["hospital_category"].isin(["Private", "Public/ Government"])]
t = pd.crosstab(cat["hospital_category"], cat["has_website"])
if t.shape == (2, 2):
    chi2, p, _, _ = stats.chi2_contingency(t)
    rows.append((f"Private vs Public: has website (chi-square, n={len(cat):,})", f"chi2={chi2:.1f}, Cramer's V={cramers_v(t):.3f}", p))

# 4. Private vs public: number of specialties listed (only those listing any)
sp = cat[cat["n_specialties"] > 0]
a = sp.loc[sp["hospital_category"] == "Private", "n_specialties"]
b = sp.loc[sp["hospital_category"] != "Private", "n_specialties"]
if len(a) > 10 and len(b) > 10:
    U, p = stats.mannwhitneyu(a, b)
    rows.append((f"Private vs Public: specialties listed (Mann-Whitney, n={len(a)}+{len(b)})",
                 f"median {a.median():.0f} vs {b.median():.0f}", p))

# 5. Do states with more records publish better data? (Spearman, states with >= 50 records)
st = df.groupby("state_clean").agg(records=("contact_score", "size"), avg_contact=("contact_score", "mean"))
st = st[st["records"] >= 50]
rho, p = stats.spearmanr(st["records"], st["avg_contact"])
rows.append((f"State size vs average contact score (Spearman, {len(st)} states)", f"rho={rho:.2f}", p))

tests = pd.DataFrame(rows, columns=["Test", "Statistic / effect size", "p-value"])
tests["Conclusion (5% level)"] = tests["p-value"].apply(verdict)
tests["p-value"] = tests["p-value"].map(fmt_p)
display(tests)

**How to read this table:** "significant" means the pattern is unlikely to be random chance. It does **not** mean the difference is large or important – check the effect size. "NOT significant" means we cannot rule out that the difference is just random variation. *(Write one sentence about each test in your own words.)*

## 26. Key insights (generated from this dataset)

The next cell calculates each statement directly from the data, so the numbers are not invented. Read the output, then **rewrite the insights in your own words** in the cell after it.

In [ ]:
ins = []
n = len(df)

# 1 Geographic concentration
sc = df["state_clean"].value_counts()
top5 = sc.head(5)
ins.append(f"1. GEOGRAPHY: {sc.index[0]} has the most hospital records ({sc.iloc[0]:,}, {sc.iloc[0]/n:.1%}). "
           f"The top 5 states ({', '.join(top5.index)}) hold {top5.sum()/n:.1%} of all {n:,} records, "
           f"while the 10 smallest states/UTs together hold only {sc.tail(10).sum()/n:.1%}.")

# 2 Missing descriptive data
known_cat = df["hospital_category"].notna().mean()
known_type = df["hospital_care_type"].notna().mean()
ins.append(f"2. DATA GAPS: only {known_cat:.1%} of records state whether the hospital is private or public, "
           f"{known_type:.1%} give a care type, and just {df['reported_beds'].notna().sum()} records ({df['reported_beds'].notna().mean():.2%}) "
           f"report a plausible number of beds. Doctor counts are reported by almost no one. Results about beds, doctors or category describe only a tiny subset.")

# 3 Contact information
ins.append(f"3. CONTACT DETAILS: {df['has_phone'].mean():.1%} of records have a phone number, {df['has_website'].mean():.1%} a website, "
           f"{df['has_email'].mean():.1%} an e-mail and {df['has_mobile'].mean():.1%} a mobile number. "
           f"{(df['contact_score'] == 0).mean():.1%} have none of these four.")

# 4 Regional differences in data quality
rg = df.groupby("region")["has_phone"].mean().sort_values()
t = pd.crosstab(df["region"], df["has_phone"])
chi2, p_reg, _, _ = stats.chi2_contingency(t)
ins.append(f"4. REGIONS: the share of records with a phone number is highest in the {rg.index[-1]} region ({rg.iloc[-1]:.1%}) and lowest in the "
           f"{rg.index[0]} region ({rg.iloc[0]:.1%}). The regional difference is {verdict(p_reg)} (chi-square p {fmt_p(p_reg) if p_reg < 0.001 else '= ' + fmt_p(p_reg)}, Cramer's V = {cramers_v(t):.2f}).")

# 5 Location data
big = df.groupby("state_clean").agg(records=("has_coordinates", "size"), coord=("has_coordinates", "mean"))
big = big[big["records"] >= 100].sort_values("coord")
ins.append(f"5. LOCATION DATA: {df['has_coordinates'].mean():.1%} of records have usable map coordinates. Among states with at least 100 records, "
           f"coverage is best in {big.index[-1]} ({big['coord'].iloc[-1]:.0%}) and worst in {big.index[0]} ({big['coord'].iloc[0]:.0%}), so maps based on this file under-represent some states.")

# 6 Facility type
ft = df["facility_type"].value_counts()
ins.append(f"6. FACILITY TYPES (from names): the most common type is '{ft.index[0]}' ({ft.iloc[0]/n:.1%}), followed by '{ft.index[1]}' ({ft.iloc[1]/n:.1%}). "
           f"{ft.get('Other', 0)/n:.1%} of names could not be classified.")

# 7 Private vs public among the hospitals that report it
cat = df[df["hospital_category"].isin(["Private", "Public/ Government"])]
cc = cat["hospital_category"].value_counts()
web = cat.groupby("hospital_category")["has_website"].mean()
ins.append(f"7. PRIVATE VS PUBLIC (only {len(cat):,} records report it): {cc.get('Private', 0):,} are private and {cc.get('Public/ Government', 0):,} public/government. "
           f"Website availability is {web.get('Private', float('nan')):.1%} for private and {web.get('Public/ Government', float('nan')):.1%} for public hospitals.")

# 8 Reported beds
b = df["reported_beds"].dropna()
ins.append(f"8. BEDS: among the {len(b)} hospitals with plausible bed counts, the median is {b.median():.0f} beds and the largest reports {b.max():.0f}. "
           f"{int((df['total_num_beds'] > 5000).sum())} impossible values (above 5,000 beds) were removed as data-entry errors.")

for s in ins:
    print(s, end="\n\n")

### Your interpretation (write this in your own words)

*Read the generated findings above, then replace the prompts below with your own sentences. A good insight has a finding, the number that supports it and why it matters.*

1. **Geography –** *Why do you think a few states hold so many records?*
2. **Data gaps –** *What does it mean that most hospitals have no category, beds or doctors? What can and can't we conclude?*
3. **Contact details –** *Can a patient find most hospitals in this directory?*
4. **Regions –** *Is data quality equal across the country?*
5. **Location data –** *What would be misleading about a map built from this file?*
6. **Facility types –** *What kind of facilities dominate India's directory?*

## 27. Conclusion and recommendations

Use the actual outputs above to complete this section in your own words.

### Suggested conclusion structure
1. State how many records and columns were analysed (30,273 hospital records and 48 columns).
2. Describe the pattern of hospital records across states and regions.
3. Explain the data-quality picture: placeholder zeros, missing category/beds/doctors, contact and coordinate coverage.
4. Summarise what the tests showed (regional differences, private vs public).
5. State the limitations: this is a directory, so counts are records not true hospital numbers, and nothing here measures quality of care.

### Recommendations
- Make category, care type, beds and contact details compulsory fields when a hospital is registered.
- Replace `0` placeholders with a real "not reported" value so that missing data is not confused with a real zero.
- Validate fields at entry (state names, impossible bed counts, pincodes).
- Compare the directory with other official health datasets (e.g. NHP / census population) before using it for planning, such as hospitals per 100,000 people.

## 28. Save a cleaned analysis copy (optional)

The original CSV remains unchanged. This optional cell saves a cleaned copy beside the notebook. Upload the original source CSV as required, and upload this cleaned copy only if your instructor wants it.


In [ ]:
# Optional: uncomment the next line to save the cleaned working data (the original CSV stays unchanged).
# df.to_csv("hospital_directory_cleaned.csv", index=False)
print("Notebook analysis complete. Save the notebook with Ctrl+S.")